# Notebook 08 — Experimental Approaches
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

**Purpose:** Test three additional approaches suggested by supervisor to see if they
improve on the current best results (Model D direct: 0.6690 Macro F1,
Uncertainty router: 0.6796 Macro F1).

**Experiments:**
1. Ensemble Stacking — unique feature sets per base model + meta-learner (LR and CatBoost)
2. SMOTE Variants — Borderline-SMOTE and ADASYN vs current class_weight baseline
3. Calibrated Confidence Router — Platt scaling to fix confidence router failure

**Important:** This is a test notebook only.
No files are saved to disk. No existing models or CSVs are modified.
All results remain inside this notebook.

**Inputs (read-only):**
- `data/processed/games_features_t4.csv`
- `data/processed/sbert_embeddings_pca50.npy`

---
## Contents
1. Setup and Imports
2. Configuration and Baselines
3. Load Data
4. Feature Sets and Train/Test Split
5. Experiment 1 — Ensemble Stacking
6. Experiment 2 — SMOTE Variants
7. Experiment 3 — Calibrated Confidence Router
8. Overall Summary

## 1. Setup and Imports

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (f1_score, roc_auc_score, accuracy_score,
                              classification_report)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE, ADASYN

print('All imports OK')

All imports OK


## 2. Configuration and Baselines

In [14]:
# ── Shared constants (must match NB03-07 exactly) ──────────────────────────
RANDOM_STATE   = 42
TEST_SIZE      = 0.20
CV_FOLDS       = 5
TARGET_COL     = 'reception_label'
DESC_COL       = 'short_description'
PCA_COMPONENTS = 50

# ── CatBoost params (depth=8, lr=0.05, iterations=1000 — agreed in NB03) ────
CB_PARAMS = {
    'depth': 8,
    'learning_rate': 0.05,
    'iterations': 1000,
    'l2_leaf_reg': 3,
    'scale_pos_weight': 0.5,
    'random_seed': RANDOM_STATE,
    'verbose': 0,
    'allow_writing_files': False
}

# ── CatBoost params WITHOUT class weight — used for SMOTE experiments ────────
# SMOTE handles imbalance at data level so scale_pos_weight is removed
CB_NO_WEIGHT = {
    'depth': 8,
    'learning_rate': 0.05,
    'iterations': 1000,
    'l2_leaf_reg': 3,
    'random_seed': RANDOM_STATE,
    'verbose': 0,
    'allow_writing_files': False 
}

# ── Baselines from NB04 FINAL (for comparison in summary) ───────────────────
BASELINE_DIRECT_F1  = 0.6690   # Model D, direct routing, test set
BASELINE_DIRECT_ACC = 0.7385
BASELINE_ROUTER_F1  = 0.6796   # Uncertainty router (D+E), test set
BASELINE_ROUTER_ACC = 0.7500

PROJECT_ROOT = Path.cwd().parent
print(f'Project root : {PROJECT_ROOT}')
print(f'CatBoost     : depth={CB_PARAMS["depth"]}, lr={CB_PARAMS["learning_rate"]}, '
      f'iterations={CB_PARAMS["iterations"]}, l2={CB_PARAMS["l2_leaf_reg"]}')
print(f'Baseline F1  : direct={BASELINE_DIRECT_F1}, router={BASELINE_ROUTER_F1}')

Project root : C:\Users\3214h\Documents\fyp-steam-reception
CatBoost     : depth=8, lr=0.05, iterations=1000, l2=3
Baseline F1  : direct=0.669, router=0.6796


## 3. Load Data

In [15]:
# ── Load structured feature dataset ─────────────────────────────────────────
df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'games_features_t4.csv')
print(f'Dataset      : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Class balance: {df[TARGET_COL].value_counts(normalize=True).round(3).to_dict()}')

# ── Load pre-computed SBERT PCA50 embeddings ─────────────────────────────────
emb_path   = PROJECT_ROOT / 'data' / 'processed' / 'sbert_embeddings_pca50.npy'
embeddings = np.load(emb_path)
print(f'Embeddings   : {embeddings.shape}  (should be {len(df)} x {PCA_COMPONENTS})')
assert embeddings.shape[0] == len(df), 'Embedding row count must match dataset'
print('Embeddings aligned with dataset ✓')

Dataset      : 20,383 rows x 56 columns
Class balance: {1: 0.718, 0: 0.282}
Embeddings   : (20383, 50)  (should be 20383 x 50)
Embeddings aligned with dataset ✓


## 4. Feature Sets and Train/Test Split

For Experiment 1 (stacking), each model trains on its **unique** features only.
Model A and B are merged since B only adds 2 features.

| Stack model | Unique features | Count |
|---|---|---|
| Model AB (merged) | price, age, genres, is_free, price_to_dlc_ratio | 15 |
| Model C | top 20 tags + 12 Steam categories | 32 |
| Model D | platform_coverage, achievements, website, languages, screenshots, trailers | 6 |
| Model E | SBERT PCA dims (text_dim_0 to text_dim_49) | 50 |

For Experiments 2 and 3, the full T4 (53 features) cumulative set is used.

In [16]:
# ── Derive feature column groups programmatically ───────────────────────────
EXCLUDE        = [TARGET_COL, DESC_COL]
ALL_STRUCTURED = [c for c in df.columns if c not in EXCLUDE]

T1_FEATURES = sorted([c for c in ALL_STRUCTURED
                       if c in ('price', 'required_age', 'genre_concentration')
                       or c.startswith('genre_')])

T2_UNIQUE   = ['is_free', 'price_to_dlc_ratio']

T3_UNIQUE   = sorted([c for c in ALL_STRUCTURED
                       if c.startswith('tag_') or c.startswith('cat_')])

T4_UNIQUE   = ['platform_coverage', 'has_achievements', 'has_website',
               'supported_languages_count', 'screenshot_count', 'movie_count']

# Cumulative tier sets (used in Experiments 2 and 3)
T2_FULL = T1_FEATURES + T2_UNIQUE
T3_FULL = T2_FULL + T3_UNIQUE
T4_FULL = T3_FULL + T4_UNIQUE        # 53 features — the full structured set

# Unique sets for stacking (Experiment 1)
FEAT_AB = T1_FEATURES + T2_UNIQUE    # 15 features — merged Model A+B
FEAT_C  = T3_UNIQUE                  # 32 features — tags and categories only
FEAT_D  = T4_UNIQUE                  # 6  features — effort signals only
TEXT_COLS = [f'text_dim_{i}' for i in range(PCA_COMPONENTS)]  # 50 text dims

print('Feature set sizes:')
print(f'  Model AB unique (merged T1+T2) : {len(FEAT_AB):>3} features')
print(f'  Model C  unique (tags+cats)    : {len(FEAT_C):>3} features')
print(f'  Model D  unique (effort sigs)  : {len(FEAT_D):>3} features')
print(f'  Model E  unique (SBERT text)   : {len(TEXT_COLS):>3} features')
print(f'  T4 full  (Experiments 2 and 3) : {len(T4_FULL):>3} features')

# ── Train / test split — identical seed and ratio as all other notebooks ─────
y   = df[TARGET_COL].values
idx = np.arange(len(df))

(X_train_df, X_test_df,
 idx_train,  idx_test,
 y_train,    y_test) = train_test_split(
    df[T4_FULL], idx, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# ── Stacking feature matrices ─────────────────────────────────────────────────
X_tr_AB = X_train_df[FEAT_AB].values
X_te_AB = X_test_df[FEAT_AB].values

X_tr_C  = X_train_df[FEAT_C].values
X_te_C  = X_test_df[FEAT_C].values

X_tr_D  = X_train_df[FEAT_D].values
X_te_D  = X_test_df[FEAT_D].values

X_tr_E  = embeddings[idx_train]       # SBERT rows aligned by original index
X_te_E  = embeddings[idx_test]

# ── T4 full matrices (Experiments 2 and 3) ───────────────────────────────────
X_tr_T4 = X_train_df[T4_FULL].values
X_te_T4 = X_test_df[T4_FULL].values

# ── Cumulative tier matrices for calibration experiment ───────────────────────
X_tr_T1 = X_train_df[T1_FEATURES].values;   X_te_T1 = X_test_df[T1_FEATURES].values
X_tr_T2 = X_train_df[T2_FULL].values;       X_te_T2 = X_test_df[T2_FULL].values
X_tr_T3 = X_train_df[T3_FULL].values;       X_te_T3 = X_test_df[T3_FULL].values

print(f'\nTrain : {len(y_train):,} games  ({y_train.mean()*100:.1f}% positive)')
print(f'Test  : {len(y_test):,} games  ({y_test.mean()*100:.1f}% positive)')

Feature set sizes:
  Model AB unique (merged T1+T2) :  15 features
  Model C  unique (tags+cats)    :  32 features
  Model D  unique (effort sigs)  :   6 features
  Model E  unique (SBERT text)   :  50 features
  T4 full  (Experiments 2 and 3) :  53 features

Train : 16,306 games  (71.8% positive)
Test  : 4,077 games  (71.8% positive)


## 5. Experiment 1 — Ensemble Stacking

**Method:** Out-of-fold (OOF) stacking.
1. Split training set into 5 folds
2. For each fold: train each base model on the other 4 folds, predict on the held-out fold
3. Collect OOF probability predictions across all 5 folds for every base model
4. Stack the 4 OOF columns into a meta-training matrix (16306 x 4)
5. Train meta-learner on the meta-training matrix
6. Retrain all base models on the FULL training set
7. Predict on test set with each base model to build meta-test matrix (4077 x 4)
8. Predict final labels using the meta-learner on the meta-test matrix

This avoids data leakage — the meta-learner never trains on predictions
made by a model that saw that game during training.

In [17]:
# ── Helper: generate out-of-fold probability predictions ────────────────────
def get_oof_predictions(X, y, params, n_splits=CV_FOLDS):
    """
    Trains a CatBoost model on n_splits-1 folds and predicts on the held-out
    fold each time. Returns an array of shape (n_samples,) with positive-class
    probability for every training game, without any data leakage.
    """
    skf      = StratifiedKFold(n_splits=n_splits, shuffle=True,
                                random_state=RANDOM_STATE)
    oof_pred = np.zeros(len(y))

    for fold_num, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        model = CatBoostClassifier(**params)
        model.fit(X[tr_idx], y[tr_idx])
        oof_pred[val_idx] = model.predict_proba(X[val_idx])[:, 1]
        print(f'    Fold {fold_num}/{n_splits} done', end='\r')

    print()
    return oof_pred

print('Generating out-of-fold predictions for each base model...')
print('(4 base models x 5 folds = 20 CatBoost training runs)')
print()

print('Model AB (merged T1+T2, 15 features):')
oof_AB = get_oof_predictions(X_tr_AB, y_train, CB_PARAMS)
print(f'  OOF AUC: {roc_auc_score(y_train, oof_AB):.4f}')

print('Model C  (tags + categories, 32 features):')
oof_C  = get_oof_predictions(X_tr_C, y_train, CB_PARAMS)
print(f'  OOF AUC: {roc_auc_score(y_train, oof_C):.4f}')

print('Model D  (effort signals, 6 features):')
oof_D  = get_oof_predictions(X_tr_D, y_train, CB_PARAMS)
print(f'  OOF AUC: {roc_auc_score(y_train, oof_D):.4f}')

print('Model E  (SBERT text dims, 50 features):')
oof_E  = get_oof_predictions(X_tr_E, y_train, CB_PARAMS)
print(f'  OOF AUC: {roc_auc_score(y_train, oof_E):.4f}')

# Stack OOF columns into meta-train matrix
meta_train = np.column_stack([oof_AB, oof_C, oof_D, oof_E])
print(f'\nMeta-train matrix: {meta_train.shape}')
print('Columns: [P(AB), P(C), P(D), P(E)] — positive-class probability per model')

Generating out-of-fold predictions for each base model...
(4 base models x 5 folds = 20 CatBoost training runs)

Model AB (merged T1+T2, 15 features):
    Fold 5/5 done
  OOF AUC: 0.6358
Model C  (tags + categories, 32 features):
    Fold 5/5 done
  OOF AUC: 0.6844
Model D  (effort signals, 6 features):
    Fold 5/5 done
  OOF AUC: 0.5827
Model E  (SBERT text dims, 50 features):
    Fold 5/5 done
  OOF AUC: 0.6363

Meta-train matrix: (16306, 4)
Columns: [P(AB), P(C), P(D), P(E)] — positive-class probability per model


In [18]:
# ── Train final base models on FULL training set ─────────────────────────────
print('Training final base models on full training set...')
print()

final_AB = CatBoostClassifier(**CB_PARAMS)
final_AB.fit(X_tr_AB, y_train)
test_prob_AB = final_AB.predict_proba(X_te_AB)[:, 1]
print(f'Model AB test AUC: {roc_auc_score(y_test, test_prob_AB):.4f}')

final_C  = CatBoostClassifier(**CB_PARAMS)
final_C.fit(X_tr_C, y_train)
test_prob_C  = final_C.predict_proba(X_te_C)[:, 1]
print(f'Model C  test AUC: {roc_auc_score(y_test, test_prob_C):.4f}')

final_D_stk = CatBoostClassifier(**CB_PARAMS)
final_D_stk.fit(X_tr_D, y_train)
test_prob_D  = final_D_stk.predict_proba(X_te_D)[:, 1]
print(f'Model D  test AUC: {roc_auc_score(y_test, test_prob_D):.4f}')

final_E_stk = CatBoostClassifier(**CB_PARAMS)
final_E_stk.fit(X_tr_E, y_train)
test_prob_E  = final_E_stk.predict_proba(X_te_E)[:, 1]
print(f'Model E  test AUC: {roc_auc_score(y_test, test_prob_E):.4f}')

# Stack test predictions into meta-test matrix
meta_test = np.column_stack([test_prob_AB, test_prob_C, test_prob_D, test_prob_E])
print(f'\nMeta-test matrix: {meta_test.shape}')

Training final base models on full training set...

Model AB test AUC: 0.6274
Model C  test AUC: 0.6981
Model D  test AUC: 0.5731
Model E  test AUC: 0.6185

Meta-test matrix: (4077, 4)


### 5a. Meta-Learner 1 — Logistic Regression

In [19]:
# ── Meta-learner 1: Logistic Regression ─────────────────────────────────────
# LR needs scaled inputs since the 4 probability columns may have different spreads
scaler          = StandardScaler()
meta_tr_scaled  = scaler.fit_transform(meta_train)
meta_te_scaled  = scaler.transform(meta_test)

lr_meta = LogisticRegression(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    max_iter=1000
)
lr_meta.fit(meta_tr_scaled, y_train)

lr_pred  = lr_meta.predict(meta_te_scaled)
lr_prob  = lr_meta.predict_proba(meta_te_scaled)[:, 1]
lr_macro = f1_score(y_test, lr_pred, average='macro')
lr_minor = f1_score(y_test, lr_pred, pos_label=0)
lr_acc   = accuracy_score(y_test, lr_pred)
lr_auc   = roc_auc_score(y_test, lr_prob)

print('Stacking with Logistic Regression meta-learner:')
print(f'  Macro F1    : {lr_macro:.4f}')
print(f'  Minority F1 : {lr_minor:.4f}')
print(f'  Accuracy    : {lr_acc:.4f}')
print(f'  AUC-ROC     : {lr_auc:.4f}')
print()
print('Meta-learner weights (how much it trusts each base model):')
for name, coef in zip(['Model AB', 'Model C', 'Model D', 'Model E'],
                       lr_meta.coef_[0]):
    print(f'  {name} : {coef:+.4f}')

Stacking with Logistic Regression meta-learner:
  Macro F1    : 0.6216
  Minority F1 : 0.5114
  Accuracy    : 0.6537
  AUC-ROC     : 0.7183

Meta-learner weights (how much it trusts each base model):
  Model AB : +0.2734
  Model C : +0.4968
  Model D : +0.1882
  Model E : +0.2896


### 5b. Meta-Learner 2 — CatBoost (Lightweight)

In [20]:
# ── Meta-learner 2: Lightweight CatBoost ─────────────────────────────────────
# Shallow tree — the meta-features are only 4 probability columns
CB_META_PARAMS = {
    'depth': 2,
    'learning_rate': 0.05,
    'iterations': 200,
    'l2_leaf_reg': 3,
    'scale_pos_weight': 0.5,
    'random_seed': RANDOM_STATE,
    'verbose': 0,
    'allow_writing_files': False
}

cb_meta = CatBoostClassifier(**CB_META_PARAMS)
cb_meta.fit(meta_train, y_train)

cb_pred  = cb_meta.predict(meta_test)
cb_prob  = cb_meta.predict_proba(meta_test)[:, 1]
cb_macro = f1_score(y_test, cb_pred, average='macro')
cb_minor = f1_score(y_test, cb_pred, pos_label=0)
cb_acc   = accuracy_score(y_test, cb_pred)
cb_auc   = roc_auc_score(y_test, cb_prob)

print('Stacking with CatBoost meta-learner:')
print(f'  Macro F1    : {cb_macro:.4f}')
print(f'  Minority F1 : {cb_minor:.4f}')
print(f'  Accuracy    : {cb_acc:.4f}')
print(f'  AUC-ROC     : {cb_auc:.4f}')
print()
print('Meta-learner feature importances:')
fi = cb_meta.get_feature_importance()
for name, imp in zip(['Model AB', 'Model C', 'Model D', 'Model E'], fi):
    print(f'  {name} : {imp:.2f}%')

Stacking with CatBoost meta-learner:
  Macro F1    : 0.6484
  Minority F1 : 0.5154
  Accuracy    : 0.6988
  AUC-ROC     : 0.7163

Meta-learner feature importances:
  Model AB : 19.03%
  Model C : 51.52%
  Model D : 7.86%
  Model E : 21.60%


### 5c. Experiment 1 Summary

In [21]:
print('=== EXPERIMENT 1 SUMMARY: ENSEMBLE STACKING ===')
print()
print(f'  {"Approach":<38} {"Macro F1":>10} {"Minor F1":>10} {"Accuracy":>10} {"AUC":>8}')
print('  ' + '-' * 80)
print(f'  {"Baseline: Model D direct":38} {BASELINE_DIRECT_F1:>10.4f} {"(see NB04)":>10} {BASELINE_DIRECT_ACC:>10.4f} {"(see NB04)":>8}')
print(f'  {"Baseline: Uncertainty router":38} {BASELINE_ROUTER_F1:>10.4f} {"(see NB04)":>10} {BASELINE_ROUTER_ACC:>10.4f} {"(see NB04)":>8}')
print(f'  {"Stacking — LR meta-learner":38} {lr_macro:>10.4f} {lr_minor:>10.4f} {lr_acc:>10.4f} {lr_auc:>8.4f}')
print(f'  {"Stacking — CatBoost meta-learner":38} {cb_macro:>10.4f} {cb_minor:>10.4f} {cb_acc:>10.4f} {cb_auc:>8.4f}')
print()
best_stack = max(lr_macro, cb_macro)
print(f'  Best stacking result : {best_stack:.4f} Macro F1')
print(f'  vs Model D direct    : {best_stack - BASELINE_DIRECT_F1:+.4f}')
print(f'  vs Uncertainty router: {best_stack - BASELINE_ROUTER_F1:+.4f}')

=== EXPERIMENT 1 SUMMARY: ENSEMBLE STACKING ===

  Approach                                 Macro F1   Minor F1   Accuracy      AUC
  --------------------------------------------------------------------------------
  Baseline: Model D direct                   0.6690 (see NB04)     0.7385 (see NB04)
  Baseline: Uncertainty router               0.6796 (see NB04)     0.7500 (see NB04)
  Stacking — LR meta-learner                 0.6216     0.5114     0.6537   0.7183
  Stacking — CatBoost meta-learner           0.6484     0.5154     0.6988   0.7163

  Best stacking result : 0.6484 Macro F1
  vs Model D direct    : -0.0206
  vs Uncertainty router: -0.0312


## 6. Experiment 2 — SMOTE Variants

**Method:** Replace `scale_pos_weight=0.5` with synthetic oversampling of the
minority class before training. The model then trains on balanced data without
needing a class weight parameter.

Two variants tested:
- **Borderline-SMOTE** — only generates synthetic samples near the decision boundary
  (the hardest cases), leaving clear examples untouched
- **ADASYN** — generates more synthetic samples in regions where the current model
  struggles most, adapting to the model's errors

Baseline for comparison: Model D with `scale_pos_weight=0.5` at Test Macro F1 = 0.6690

In [22]:
results_smote = {}

# ── Borderline-SMOTE ─────────────────────────────────────────────────────────
print('Running Borderline-SMOTE...')
bsmote          = BorderlineSMOTE(random_state=RANDOM_STATE, kind='borderline-1')
X_bsmote, y_bsmote = bsmote.fit_resample(X_tr_T4, y_train)
print(f'  Before: {len(y_train):,} games  ({y_train.mean()*100:.1f}% positive)')
print(f'  After : {len(y_bsmote):,} games  ({y_bsmote.mean()*100:.1f}% positive)')

model_bsmote = CatBoostClassifier(**CB_NO_WEIGHT)
model_bsmote.fit(X_bsmote, y_bsmote)
pred_bsmote = model_bsmote.predict(X_te_T4)
prob_bsmote = model_bsmote.predict_proba(X_te_T4)[:, 1]

results_smote['Borderline-SMOTE'] = {
    'macro': f1_score(y_test, pred_bsmote, average='macro'),
    'minor': f1_score(y_test, pred_bsmote, pos_label=0),
    'acc'  : accuracy_score(y_test, pred_bsmote),
    'auc'  : roc_auc_score(y_test, prob_bsmote)
}
r = results_smote['Borderline-SMOTE']
print(f'  Macro F1: {r["macro"]:.4f}  Minority F1: {r["minor"]:.4f}  '
      f'Accuracy: {r["acc"]:.4f}  AUC: {r["auc"]:.4f}')
print()

# ── ADASYN ───────────────────────────────────────────────────────────────────
print('Running ADASYN...')
adasyn          = ADASYN(random_state=RANDOM_STATE)
X_adasyn, y_adasyn = adasyn.fit_resample(X_tr_T4, y_train)
print(f'  Before: {len(y_train):,} games  ({y_train.mean()*100:.1f}% positive)')
print(f'  After : {len(y_adasyn):,} games  ({y_adasyn.mean()*100:.1f}% positive)')

model_adasyn = CatBoostClassifier(**CB_NO_WEIGHT)
model_adasyn.fit(X_adasyn, y_adasyn)
pred_adasyn = model_adasyn.predict(X_te_T4)
prob_adasyn = model_adasyn.predict_proba(X_te_T4)[:, 1]

results_smote['ADASYN'] = {
    'macro': f1_score(y_test, pred_adasyn, average='macro'),
    'minor': f1_score(y_test, pred_adasyn, pos_label=0),
    'acc'  : accuracy_score(y_test, pred_adasyn),
    'auc'  : roc_auc_score(y_test, prob_adasyn)
}
r = results_smote['ADASYN']
print(f'  Macro F1: {r["macro"]:.4f}  Minority F1: {r["minor"]:.4f}  '
      f'Accuracy: {r["acc"]:.4f}  AUC: {r["auc"]:.4f}')

Running Borderline-SMOTE...
  Before: 16,306 games  (71.8% positive)
  After : 23,410 games  (50.0% positive)
  Macro F1: 0.6514  Minority F1: 0.4604  Accuracy: 0.7562  AUC: 0.7433

Running ADASYN...
  Before: 16,306 games  (71.8% positive)
  After : 22,557 games  (51.9% positive)
  Macro F1: 0.6546  Minority F1: 0.4659  Accuracy: 0.7577  AUC: 0.7437


### 6a. Experiment 2 Summary

In [23]:
print('=== EXPERIMENT 2 SUMMARY: SMOTE VARIANTS ===')
print()
print(f'  {"Approach":<35} {"Macro F1":>10} {"Minor F1":>10} {"Accuracy":>10} {"AUC":>8}')
print('  ' + '-' * 78)
print(f'  {"Baseline (scale_pos_weight=0.5)":<35} {BASELINE_DIRECT_F1:>10.4f} {"(see NB04)":>10} {BASELINE_DIRECT_ACC:>10.4f} {"(see NB04)":>8}')
for name, r in results_smote.items():
    diff = r['macro'] - BASELINE_DIRECT_F1
    print(f'  {name:<35} {r["macro"]:>10.4f} {r["minor"]:>10.4f} {r["acc"]:>10.4f} {r["auc"]:>8.4f}')
print()
best_smote = max(results_smote.values(), key=lambda r: r['macro'])
print(f'  Best SMOTE result : {best_smote["macro"]:.4f} Macro F1')
print(f'  vs baseline       : {best_smote["macro"] - BASELINE_DIRECT_F1:+.4f}')

=== EXPERIMENT 2 SUMMARY: SMOTE VARIANTS ===

  Approach                              Macro F1   Minor F1   Accuracy      AUC
  ------------------------------------------------------------------------------
  Baseline (scale_pos_weight=0.5)         0.6690 (see NB04)     0.7385 (see NB04)
  Borderline-SMOTE                        0.6514     0.4604     0.7562   0.7433
  ADASYN                                  0.6546     0.4659     0.7577   0.7437

  Best SMOTE result : 0.6546 Macro F1
  vs baseline       : -0.0144


## 7. Experiment 3 — Calibrated Confidence Router

**Background:** In NB04 the confidence router failed badly at T3 and T4.
The root cause: simpler models with fewer features are artificially certain —
not because they know more, but because they have fewer dimensions pulling them
in different directions. The raw probabilities are not well-calibrated.

**Fix — Platt scaling:** After training each model, fit a small sigmoid
(Logistic Regression) on top of its raw probability outputs using 3-fold CV.
This rescales probabilities so they reflect true likelihoods.
A calibrated model that outputs 80% should be right ~80% of the time.

Cumulative feature sets are used here (not unique), matching the original
NB04 confidence router setup for a fair comparison.

In [24]:
# ── Train and calibrate Models A-D on cumulative feature sets ───────────────
TIER_DATA = {
    'A': (X_tr_T1, X_te_T1, 'T1 (13 features)'),
    'B': (X_tr_T2, X_te_T2, 'T2 (15 features)'),
    'C': (X_tr_T3, X_te_T3, 'T3 (47 features)'),
    'D': (X_tr_T4, X_te_T4, 'T4 (53 features)'),
}

raw_models_calib = {}
cal_models_calib = {}

print('Training and calibrating Models A-D...')
print()

for tier_name, (X_tr, X_te, label) in TIER_DATA.items():
    print(f'Model {tier_name} — {label}:')

    # Train raw model
    raw_m = CatBoostClassifier(**CB_PARAMS)
    raw_m.fit(X_tr, y_train)
    raw_probs = raw_m.predict_proba(X_te)[:, 1]
    raw_models_calib[tier_name] = (raw_m, X_te)

    # Calibrate using Platt scaling (sigmoid) with 3-fold CV
    cal_m = CalibratedClassifierCV(
        CatBoostClassifier(**CB_PARAMS),
        method='sigmoid',
        cv=3
    )
    cal_m.fit(X_tr, y_train)
    cal_probs = cal_m.predict_proba(X_te)[:, 1]
    cal_models_calib[tier_name] = (cal_m, X_te)

    print(f'  Raw probs       — mean: {raw_probs.mean():.4f}  std: {raw_probs.std():.4f}')
    print(f'  Calibrated probs — mean: {cal_probs.mean():.4f}  std: {cal_probs.std():.4f}')
    print()

Training and calibrating Models A-D...

Model A — T1 (13 features):
  Raw probs       — mean: 0.5885  std: 0.1803
  Calibrated probs — mean: 0.7158  std: 0.0842

Model B — T2 (15 features):
  Raw probs       — mean: 0.5983  std: 0.1903
  Calibrated probs — mean: 0.7161  std: 0.0885

Model C — T3 (47 features):
  Raw probs       — mean: 0.6454  std: 0.2426
  Calibrated probs — mean: 0.7159  std: 0.1512

Model D — T4 (53 features):
  Raw probs       — mean: 0.6612  std: 0.2446
  Calibrated probs — mean: 0.7174  std: 0.1610



In [25]:
# ── Confidence router function ────────────────────────────────────────────────
def run_confidence_router(models_dict, y_true, label=''):
    """
    For each test game, query all models and pick the one with highest
    confidence (max distance from 0.5). Returns Macro F1, Minority F1,
    Accuracy, and a dict of how many times each model was chosen.
    """
    tier_names  = list(models_dict.keys())
    n_games     = len(y_true)
    n_models    = len(tier_names)
    all_probs   = np.zeros((n_games, n_models))
    confidence  = np.zeros((n_games, n_models))

    for i, (name, (model, X_te)) in enumerate(models_dict.items()):
        probs            = model.predict_proba(X_te)[:, 1]
        all_probs[:, i]  = probs
        confidence[:, i] = np.abs(probs - 0.5)

    best_idx    = np.argmax(confidence, axis=1)
    final_preds = np.zeros(n_games, dtype=int)
    usage       = {name: 0 for name in tier_names}

    for game_i in range(n_games):
        chosen      = tier_names[best_idx[game_i]]
        prob        = all_probs[game_i, best_idx[game_i]]
        final_preds[game_i] = 1 if prob >= 0.5 else 0
        usage[chosen] += 1

    macro = f1_score(y_true, final_preds, average='macro')
    minor = f1_score(y_true, final_preds, pos_label=0)
    acc   = accuracy_score(y_true, final_preds)

    if label:
        print(f'{label}:')
        print(f'  Macro F1    : {macro:.4f}')
        print(f'  Minority F1 : {minor:.4f}')
        print(f'  Accuracy    : {acc:.4f}')
        print(f'  Model usage : {usage}')
        print()

    return macro, minor, acc, usage

print('=== CONFIDENCE ROUTER COMPARISON ===')
print()

raw_macro, raw_minor, raw_acc, raw_usage = run_confidence_router(
    raw_models_calib, y_test, 'Raw confidence router (uncalibrated)')

cal_macro, cal_minor, cal_acc, cal_usage = run_confidence_router(
    cal_models_calib, y_test, 'Calibrated confidence router (Platt scaling)')

=== CONFIDENCE ROUTER COMPARISON ===

Raw confidence router (uncalibrated):
  Macro F1    : 0.6496
  Minority F1 : 0.4800
  Accuracy    : 0.7317
  Model usage : {'A': 547, 'B': 531, 'C': 1353, 'D': 1646}

Calibrated confidence router (Platt scaling):
  Macro F1    : 0.5492
  Minority F1 : 0.2504
  Accuracy    : 0.7474
  Model usage : {'A': 761, 'B': 495, 'C': 1183, 'D': 1638}



### 7a. Experiment 3 Summary

In [26]:
print('=== EXPERIMENT 3 SUMMARY: CALIBRATED CONFIDENCE ROUTER ===')
print()
print(f'  {"Approach":<45} {"Macro F1":>10} {"Minor F1":>10} {"Accuracy":>10}')
print('  ' + '-' * 80)
print(f'  {"Baseline: Direct routing (Model D)":<45} {BASELINE_DIRECT_F1:>10.4f} {"(see NB04)":>10} {BASELINE_DIRECT_ACC:>10.4f}')
print(f'  {"NB04 confidence router (from results)":<45} {"0.6462":>10} {"—":>10} {"—":>10}')
print(f'  {"Raw confidence router (this run)":<45} {raw_macro:>10.4f} {raw_minor:>10.4f} {raw_acc:>10.4f}')
print(f'  {"Calibrated confidence router":<45} {cal_macro:>10.4f} {cal_minor:>10.4f} {cal_acc:>10.4f}')
print()
if cal_macro > raw_macro:
    print(f'  Calibration improved confidence routing by {cal_macro - raw_macro:+.4f} Macro F1')
else:
    print(f'  Calibration did NOT improve confidence routing ({cal_macro - raw_macro:+.4f} Macro F1)')
if cal_macro > BASELINE_DIRECT_F1:
    print(f'  Calibrated router BEATS direct routing by {cal_macro - BASELINE_DIRECT_F1:+.4f}')
else:
    print(f'  Calibrated router still BELOW direct routing by {cal_macro - BASELINE_DIRECT_F1:+.4f}')

=== EXPERIMENT 3 SUMMARY: CALIBRATED CONFIDENCE ROUTER ===

  Approach                                        Macro F1   Minor F1   Accuracy
  --------------------------------------------------------------------------------
  Baseline: Direct routing (Model D)                0.6690 (see NB04)     0.7385
  NB04 confidence router (from results)             0.6462          —          —
  Raw confidence router (this run)                  0.6496     0.4800     0.7317
  Calibrated confidence router                      0.5492     0.2504     0.7474

  Calibration did NOT improve confidence routing (-0.1004 Macro F1)
  Calibrated router still BELOW direct routing by -0.1198


## 8. Overall Summary

In [27]:
print('=' * 82)
print('NB08 OVERALL SUMMARY — ALL EXPERIMENTS vs BASELINES')
print('=' * 82)
print()
print(f'  {"Approach":<42} {"Macro F1":>10} {"Accuracy":>10} {"vs Direct":>10}')
print('  ' + '-' * 76)

all_results = [
    ('Baseline: Model D direct routing',        BASELINE_DIRECT_F1, BASELINE_DIRECT_ACC,  None),
    ('Baseline: Uncertainty router (D+E)',       BASELINE_ROUTER_F1, BASELINE_ROUTER_ACC,  None),
    ('Stacking: LR meta-learner',               lr_macro,           lr_acc,               lr_macro - BASELINE_DIRECT_F1),
    ('Stacking: CatBoost meta-learner',         cb_macro,           cb_acc,               cb_macro - BASELINE_DIRECT_F1),
    ('SMOTE: Borderline-SMOTE',                 results_smote['Borderline-SMOTE']['macro'],
                                                results_smote['Borderline-SMOTE']['acc'],
                                                results_smote['Borderline-SMOTE']['macro'] - BASELINE_DIRECT_F1),
    ('SMOTE: ADASYN',                           results_smote['ADASYN']['macro'],
                                                results_smote['ADASYN']['acc'],
                                                results_smote['ADASYN']['macro'] - BASELINE_DIRECT_F1),
    ('Calibrated confidence router',            cal_macro,          cal_acc,              cal_macro - BASELINE_DIRECT_F1),
]

for name, macro, acc, diff in all_results:
    diff_str = f'{diff:+.4f}' if diff is not None else '—'
    print(f'  {name:<42} {macro:>10.4f} {acc:>10.4f} {diff_str:>10}')

print()

# Find the best new approach
new_only = [(n, m, a, d) for n, m, a, d in all_results if d is not None]
best     = max(new_only, key=lambda x: x[1])
print(f'  Best new approach    : {best[0]}')
print(f'  Best Macro F1        : {best[1]:.4f}')
print(f'  vs Model D direct    : {best[1] - BASELINE_DIRECT_F1:+.4f}')
print(f'  vs Uncertainty router: {best[1] - BASELINE_ROUTER_F1:+.4f}')
print()
print('  Recommendation:')
if best[1] > BASELINE_ROUTER_F1:
    print(f'  {best[0]} beats the current best (uncertainty router).')
    print('  Consider integrating this approach into the main system.')
elif best[1] > BASELINE_DIRECT_F1:
    print(f'  {best[0]} improves on direct routing but not the uncertainty router.')
    print('  The uncertainty router remains the best overall strategy.')
else:
    print('  None of the new approaches improved on the current baselines.')
    print('  The existing system (Model D + uncertainty router) remains optimal.')
    print('  These negative results are still valid research contributions.')

NB08 OVERALL SUMMARY — ALL EXPERIMENTS vs BASELINES

  Approach                                     Macro F1   Accuracy  vs Direct
  ----------------------------------------------------------------------------
  Baseline: Model D direct routing               0.6690     0.7385          —
  Baseline: Uncertainty router (D+E)             0.6796     0.7500          —
  Stacking: LR meta-learner                      0.6216     0.6537    -0.0474
  Stacking: CatBoost meta-learner                0.6484     0.6988    -0.0206
  SMOTE: Borderline-SMOTE                        0.6514     0.7562    -0.0176
  SMOTE: ADASYN                                  0.6546     0.7577    -0.0144
  Calibrated confidence router                   0.5492     0.7474    -0.1198

  Best new approach    : SMOTE: ADASYN
  Best Macro F1        : 0.6546
  vs Model D direct    : -0.0144
  vs Uncertainty router: -0.0250

  Recommendation:
  None of the new approaches improved on the current baselines.
  The existing system (